In [7]:
# NEW CELL — Build test_data_3.xlsx with daily IndianAPI fills using _T_date columns

import math
import time
import json
import requests
import pandas as pd
from datetime import datetime, date

# ====== CONFIG ======
INPUT_XLSX  = "Test-Data_1.xlsx"     # provided reference workbook
OUTPUT_XLSX = "test_data_3.xlsx"     # NEW workbook to create
API_KEY     = "sk-live-qRes50UXfu48WwWyW8FEavml5F6GP7kymfOeC9Wb"    # <<<<< REQUIRED
BASE_URL    = "https://stock.indianapi.in"

# Column names in the updated file
COL_TICKER = "Ticker"
COL_RDATE  = "Result Date"            # expected DD-MM-YYYY (string)
# Dates
COL_TD   = "_T_date"
COL_T1D  = "_T1_date"
COL_T2D  = "_T2_date"
COL_T3D  = "_T3_date"
# Prices
COL_TP   = "Closing Price (T)"
COL_T1P  = "Closing Price (T+1)"
COL_T2P  = "Closing Price (T+2)"
COL_T3P  = "Closing Price (T+3)"

# ====== HTTP session ======
SESSION = requests.Session()
SESSION.headers.update({"x-api-key": API_KEY})

def log(msg):
    print(msg, flush=True)

def to_date_ddmmyyyy(s):
    if pd.isna(s) or str(s).strip() == "":
        return None
    try:
        return pd.to_datetime(str(s), dayfirst=True).date()
    except Exception:
        return None

def fmt_ddmmyyyy(d):
    if d is None or (isinstance(d, float) and math.isnan(d)):
        return None
    return pd.Timestamp(d).strftime("%d-%m-%Y")

def fetch_indianapi_daily_series(stock_name, period="1yr", retry=2, sleep=0.6):
    """
    GET /historical_data?stock_name=...&period=1yr&filter=price
    Return a sorted list of (date, close) from a dataset with meta.is_weekly == False
    Prefer NSE over BSE if both exist.
    """
    last_exc = None
    for attempt in range(retry + 1):
        try:
            r = SESSION.get(
                f"{BASE_URL}/historical_data",
                params={"stock_name": stock_name, "period": period, "filter": "price"},
                timeout=20,
            )
            if r.status_code == 429:
                time.sleep(1.5 + attempt)
                continue
            r.raise_for_status()
            data = r.json()
            dsets = data.get("datasets") or []
            candidates = []
            for ds in dsets:
                meta = ds.get("meta") or {}
                if bool(meta.get("is_weekly", False)):
                    continue  # skip weekly
                label = (ds.get("label") or "").upper()
                vals  = ds.get("values") or []
                series = []
                for row in vals:
                    if not row or len(row) < 2:
                        continue
                    try:
                        dt = pd.to_datetime(row[0]).date()
                        px = float(row[1])
                        series.append((dt, px))
                    except Exception:
                        continue
                if not series:
                    continue
                series.sort(key=lambda x: x[0])
                # Score by venue preference
                score = 2 if "NSE" in label else (1 if "BSE" in label else 0)
                candidates.append((score, label, series))
            if not candidates:
                return None
            candidates.sort(key=lambda x: (-x[0], x[1]))
            return candidates[0][2]
        except Exception as e:
            last_exc = e
            if attempt < retry:
                time.sleep(sleep * (attempt + 1))
                continue
    log(f"  [warn] daily fetch failed for {stock_name}: {last_exc}")
    return None

def pick_T_T1_T2_T3(series, result_date):
    """
    T: exact trade on Result Date else strictly previous trading day.
    T+1/2/3: next 3 trading days after T.
    Returns (t_date,t_px,t1_date,t1_px,t2_date,t2_px,t3_date,t3_px) or None.
    """
    if not series:
        return None
    dates  = [d for d, _ in series]
    prices = dict(series)
    if result_date in prices:
        t_date = result_date
    else:
        before = [d for d in dates if d < result_date]
        if not before:
            return None
        t_date = before[-1]
    t_px = prices[t_date]
    after = [d for d in dates if d > t_date]
    t1d = after[0] if len(after) > 0 else None
    t2d = after[1] if len(after) > 1 else None
    t3d = after[2] if len(after) > 2 else None
    return (
        t_date, t_px,
        t1d, prices.get(t1d) if t1d else None,
        t2d, prices.get(t2d) if t2d else None,
        t3d, prices.get(t3d) if t3d else None,
    )

# ---- Load the updated input workbook ----
xls = pd.ExcelFile(INPUT_XLSX)
log(f"Workbook sheets: {xls.sheet_names}")

# Choose the first sheet containing Ticker + Result Date
base_sheet = None
for sn in xls.sheet_names:
    df_try = pd.read_excel(INPUT_XLSX, sheet_name=sn)
    if COL_TICKER in df_try.columns and COL_RDATE in df_try.columns:
        base_sheet = sn
        break
if base_sheet is None:
    raise SystemExit(f"Could not find a sheet with '{COL_TICKER}' and '{COL_RDATE}' in {INPUT_XLSX}")

df = pd.read_excel(INPUT_XLSX, sheet_name=base_sheet)
n_rows = len(df)
log(f"Selected base sheet: '{base_sheet}'  rows={n_rows}")

# Ensure the date/price columns exist
for c in [COL_TD, COL_T1D, COL_T2D, COL_T3D, COL_TP, COL_T1P, COL_T2P, COL_T3P]:
    if c not in df.columns:
        df[c] = pd.NA

# Determine rows to process:
# - If _T_date is empty OR Closing Price (T) is empty OR any pair is inconsistent (date present w/o price or price w/o date)
def is_blank(x): return (pd.isna(x) or str(x).strip() == "")

needs = []
for i, r in df.iterrows():
    t_blank  = is_blank(r[COL_TD]) or pd.isna(r[COL_TP])
    t1_incon = (not is_blank(r[COL_T1D]) and pd.isna(r[COL_T1P])) or (is_blank(r[COL_T1D]) and not pd.isna(r[COL_T1P]))
    t2_incon = (not is_blank(r[COL_T2D]) and pd.isna(r[COL_T2P])) or (is_blank(r[COL_T2D]) and not pd.isna(r[COL_T2P]))
    t3_incon = (not is_blank(r[COL_T3D]) and pd.isna(r[COL_T3P])) or (is_blank(r[COL_T3D]) and not pd.isna(r[COL_T3P]))
    if t_blank or t1_incon or t2_incon or t3_incon:
        needs.append(i)

log(f"Rows queued for price/date backfill: {len(needs)}")

# Prepare per-ticker daily cache
tickers = sorted(df.loc[needs, COL_TICKER].dropna().astype(str).unique())
log(f"Unique tickers to fetch: {len(tickers)} -> {tickers[:10]}{' ...' if len(tickers)>10 else ''}")

cache = {}
for tk in tickers:
    log(f"Fetching IndianAPI (daily only) for {tk} ...")
    s = fetch_indianapi_daily_series(tk, period="1yr")
    if s is None:
        log("  -> NO usable daily dataset (weekly/empty).")
    else:
        log(f"  -> OK: {len(s)} points  range {s[0][0]} .. {s[-1][0]}")
    cache[tk] = s
    time.sleep(0.25)

# Backfill
not_found = []
filled = 0

for i in needs:
    row = df.loc[i]
    tk = str(row[COL_TICKER]).strip()
    rd = to_date_ddmmyyyy(row[COL_RDATE])
    if rd is None:
        not_found.append({"Ticker": tk, "Result Date": row[COL_RDATE], "Reason": "Invalid Result Date"})
        continue

    s = cache.get(tk)
    if not s:
        not_found.append({"Ticker": tk, "Result Date": fmt_ddmmyyyy(rd), "Reason": "No daily series"})
        continue

    pick = pick_T_T1_T2_T3(s, rd)
    if pick is None:
        not_found.append({"Ticker": tk, "Result Date": fmt_ddmmyyyy(rd), "Reason": "No trade on/before Result Date in 1yr window"})
        continue

    t_d, t_p, t1_d, t1_p, t2_d, t2_p, t3_d, t3_p = pick

    # Fill T (both date & price if either missing)
    if is_blank(row[COL_TD]) or pd.isna(row[COL_TP]):
        df.at[i, COL_TD] = fmt_ddmmyyyy(t_d)
        df.at[i, COL_TP] = t_p

    # Fill T+1/T+2/T+3 only if either side missing (keep existing if both present)
    if is_blank(row[COL_T1D]) or pd.isna(row[COL_T1P]):
        df.at[i, COL_T1D] = fmt_ddmmyyyy(t1_d)
        df.at[i, COL_T1P] = t1_p
    if is_blank(row[COL_T2D]) or pd.isna(row[COL_T2P]):
        df.at[i, COL_T2D] = fmt_ddmmyyyy(t2_d)
        df.at[i, COL_T2P] = t2_p
    if is_blank(row[COL_T3D]) or pd.isna(row[COL_T3P]):
        df.at[i, COL_T3D] = fmt_ddmmyyyy(t3_d)
        df.at[i, COL_T3P] = t3_p

    filled += 1
    if filled <= 8:
        log(f"  [{tk}] RD={fmt_ddmmyyyy(rd)} -> "
            f"T={fmt_ddmmyyyy(t_d)} {t_p} | T+1={fmt_ddmmyyyy(t1_d)} {t1_p} | "
            f"T+2={fmt_ddmmyyyy(t2_d)} {t2_p} | T+3={fmt_ddmmyyyy(t3_d)} {t3_p}")

log(f"\nSummary: updated={filled}  not_found={len(not_found)}  total_rows={n_rows}")

# Create the new workbook with required sheets
all_prices = df.copy()
not_found_df = pd.DataFrame(not_found, columns=["Ticker", "Result Date", "Reason"])

# Ensure same row count as input
assert len(all_prices) == n_rows, "all_prices row count must equal source row count."

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl", mode="w") as xw:
    all_prices.to_excel(xw, sheet_name="all_prices", index=False)
    not_found_df.to_excel(xw, sheet_name="not_found", index=False)

log(f"Saved: {OUTPUT_XLSX}  | sheets: all_prices={len(all_prices)} rows, not_found={len(not_found_df)} rows")


Workbook sheets: ['Sheet1']
Selected base sheet: 'Sheet1'  rows=585
Rows queued for price/date backfill: 235
Unique tickers to fetch: 45 -> ['ASHOKLEY', 'AUBANK', 'AUROPHARMA', 'AXISBANK', 'BAJAJ AUTO', 'BAJFINANCE', 'BIOCON', 'BOSCHLTD', 'BRITANNIA', 'CIPLA'] ...
Fetching IndianAPI (daily only) for ASHOKLEY ...
  -> OK: 251 points  range 2024-08-13 .. 2025-08-13
Fetching IndianAPI (daily only) for AUBANK ...
  -> OK: 251 points  range 2024-08-13 .. 2025-08-13
Fetching IndianAPI (daily only) for AUROPHARMA ...
  -> OK: 251 points  range 2024-08-13 .. 2025-08-13
Fetching IndianAPI (daily only) for AXISBANK ...
  -> OK: 251 points  range 2024-08-13 .. 2025-08-13
Fetching IndianAPI (daily only) for BAJAJ AUTO ...
  -> OK: 251 points  range 2024-08-13 .. 2025-08-13
Fetching IndianAPI (daily only) for BAJFINANCE ...
  -> OK: 251 points  range 2024-08-13 .. 2025-08-13
Fetching IndianAPI (daily only) for BIOCON ...
  -> OK: 251 points  range 2024-08-13 .. 2025-08-13
Fetching IndianAPI (daily 

C:\Users\cecme\AppData\Local\Temp\ipykernel_1292\2253146797.py:41: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(str(s), dayfirst=True).date()



Summary: updated=182  not_found=53  total_rows=585
Saved: test_data_3.xlsx  | sheets: all_prices=585 rows, not_found=53 rows


In [8]:
# === Cell: Fill missing T/T+1/T+2/T+3 via Alpha Vantage (two keys) and write test_data_4.xlsx ===
import pandas as pd
import requests, time
from datetime import datetime
from pathlib import Path

# -----------------------------
# CONFIG — EDIT ONLY THESE
# -----------------------------
INPUT_XLSX  = "test_data_3.xlsx"
OUTPUT_XLSX = "test_data_4.xlsx"

# Put your two Alpha Vantage keys here
AV_KEYS = [
    "NDW9P3OPP015FRKT",  # used for first 25 missing rows (in order)
    "CSQEI87CX68SEMX6",  # used for next 25 missing rows
]

# AV free tier is ~5 calls/min per key; 13s gap is safe.
PER_KEY_MIN_GAP_SECONDS = 13

# -----------------------------
# Helpers
# -----------------------------
def _fmt_date(dt: pd.Timestamp | None) -> str:
    """Return DD-MM-YYYY or '' for None/NaT."""
    if dt is None or pd.isna(dt):
        return ""
    return dt.strftime("%d-%m-%Y")

def _parse_result_date(x) -> pd.Timestamp | None:
    """Parse 'Result Date' which is DD-MM-YYYY (or already datetime)."""
    if pd.isna(x) or x == "":
        return None
    # Accept strings or datetimes
    try:
        return pd.to_datetime(x, dayfirst=True, errors="coerce")
    except Exception:
        return None

class AVCache:
    """Cache AlphaVantage daily close series per symbol with simple per-key rate limiting."""
    def __init__(self, keys, min_gap_sec=13):
        self.keys = keys
        self.cache: dict[str, pd.Series] = {}
        self.key_for_symbol: dict[str, int] = {}  # remember which key fetched a symbol
        self.last_call_at = {i: 0.0 for i, _ in enumerate(keys)}
        self.session = requests.Session()

    def _respect_rate_limit(self, key_idx: int):
        elapsed = time.time() - self.last_call_at[key_idx]
        wait_for = PER_KEY_MIN_GAP_SECONDS - elapsed
        if wait_for > 0:
            time.sleep(wait_for)

    def fetch_series(self, symbol: str, key_idx: int) -> pd.Series:
        """Return a pandas Series indexed by datetime with close prices."""
        if symbol in self.cache:
            return self.cache[symbol]

        if key_idx >= len(self.keys):
            key_idx = len(self.keys) - 1  # clamp to last key if index exceeded

        key = self.keys[key_idx]
        if not key or key.startswith("REPLACE_WITH_KEY"):
            raise RuntimeError("Alpha Vantage key is missing. Please set AV_KEYS.")

        # Rate limit per key
        self._respect_rate_limit(key_idx)

        params = {
            "function": "TIME_SERIES_DAILY",
            "symbol": symbol,
            "outputsize": "full",
            "apikey": key,
        }
        url = "https://www.alphavantage.co/query"

        # Up to 3 attempts (handle transient limits)
        for attempt in range(3):
            r = self.session.get(url, params=params, timeout=40)
            self.last_call_at[key_idx] = time.time()
            if r.status_code != 200:
                print(f"   ! HTTP {r.status_code} for {symbol} (attempt {attempt+1})")
                time.sleep(5 * (attempt + 1))
                continue
            try:
                j = r.json()
            except Exception as e:
                print(f"   ! JSON error for {symbol}: {e} (attempt {attempt+1})")
                time.sleep(5 * (attempt + 1))
                continue

            if "Time Series (Daily)" not in j:
                # Rate-limit messages show up as 'Note' or 'Information'
                note = j.get("Note") or j.get("Information") or j.get("Error Message")
                print(f"   ! AV response issue for {symbol}: {note!r}")
                # Back-off a bit more if we hit a limit
                time.sleep(20 * (attempt + 1))
                continue

            ts = j["Time Series (Daily)"]
            # Build close-price series
            s = pd.Series(
                {pd.to_datetime(k): float(v["4. close"]) for k, v in ts.items()}
            ).sort_index()
            self.cache[symbol] = s
            self.key_for_symbol[symbol] = key_idx
            return s

        raise RuntimeError(f"Failed to fetch Alpha Vantage data for {symbol} after retries.")

def compute_T_block(series: pd.Series, result_dt: pd.Timestamp):
    """
    Given a daily close series and a Result Date, return:
    T_date, T_price, T1_date, T1_price, T2_date, T2_price, T3_date, T3_price
    Rule: T = exact Result Date if traded; else previous trading day strictly before Result Date;
          T+N = next N trading days after T.
    """
    idx = series.index

    # Exact trade on Result Date?
    if result_dt in idx:
        T_idx_pos = idx.get_indexer([result_dt])[0]
    else:
        # previous trading day strictly before Result Date
        prev_pos = idx.searchsorted(result_dt) - 1
        if prev_pos < 0:
            return (None, None, None, None, None, None, None, None)
        T_idx_pos = prev_pos

    # T
    T_date = idx[T_idx_pos]
    T_price = float(series.iloc[T_idx_pos])

    # T+1, T+2, T+3
    def _next_pos(pos):
        return pos + 1 if pos + 1 < len(idx) else None

    T1_pos = _next_pos(T_idx_pos)
    T2_pos = _next_pos(T1_pos) if T1_pos is not None else None
    T3_pos = _next_pos(T2_pos) if T2_pos is not None else None

    T1_date = idx[T1_pos] if T1_pos is not None else None
    T2_date = idx[T2_pos] if T2_pos is not None else None
    T3_date = idx[T3_pos] if T3_pos is not None else None

    T1_price = float(series.iloc[T1_pos]) if T1_pos is not None else None
    T2_price = float(series.iloc[T2_pos]) if T2_pos is not None else None
    T3_price = float(series.iloc[T3_pos]) if T3_pos is not None else None

    return (T_date, T_price, T1_date, T1_price, T2_date, T2_price, T3_date, T3_price)

# -----------------------------
# Load workbook and find work items
# -----------------------------
print(f"Reading input workbook: {INPUT_XLSX}")
xls = pd.ExcelFile(INPUT_XLSX)
sheets = {name: xls.parse(name) for name in xls.sheet_names}

if "all_prices" not in sheets:
    raise SystemExit("Sheet 'all_prices' not found in the workbook.")

df = sheets["all_prices"].copy()

required_cols = [
    "Company","Ticker","AV_Ticker.BSE","Sector","Cap Category","Period",
    "Sales","Expenses","Operating Profit","OPM %","Other Income","Interest",
    "Depreciation","Profit before tax","Tax %","Net Profit","EPS in Rs",
    "Result Date","Revenue","Financing Profit","Financing Margin %","Gross NPA %","Net NPA %",
    "_T_date","Closing Price (T)","_T1_date","Closing Price (T+1)","_T2_date","Closing Price (T+2)","_T3_date","Closing Price (T+3)"
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise SystemExit(f"Missing expected columns in 'all_prices': {missing}")

# Normalize empties for _T_date
mask_missing_T = df["_T_date"].astype(str).str.strip().isin(["", "NaT"]) | df["_T_date"].isna()
rows_to_fill = df[mask_missing_T].copy()
n_to_fill = len(rows_to_fill)

print(f"Total rows in 'all_prices': {len(df)}")
print(f"Rows needing prices (blank _T_date): {n_to_fill}")

if n_to_fill == 0:
    print("Nothing to fill. Writing out a copy as test_data_4.xlsx …")

# -----------------------------
# Fill using AV (two keys)
# -----------------------------
av = AVCache(AV_KEYS, PER_KEY_MIN_GAP_SECONDS)

exact_hits = 0
prev_hits  = 0
failures   = 0
updated    = 0

# Assign a key per row index: first 25 -> key 0, next 25 -> key 1, etc, clamped
indices_to_fill = rows_to_fill.index.tolist()
for i, ridx in enumerate(indices_to_fill, start=1):
    row = df.loc[ridx]
    sym  = str(row.get("AV_Ticker.BSE", "")).strip()
    tick = str(row.get("Ticker", "")).strip()
    rdate_raw = row.get("Result Date", "")
    rdate = _parse_result_date(rdate_raw)
    key_idx = min((i-1)//25, len(AV_KEYS)-1)

    print(f"\n[{i}/{n_to_fill}] {tick} | symbol={sym or '<blank>'} | Result Date={rdate_raw} | using key K{key_idx+1}")

    if not sym:
        print("   ! No AV_Ticker.BSE symbol. Skipping row.")
        failures += 1
        continue
    if rdate is None or pd.isna(rdate):
        print("   ! Invalid Result Date. Skipping row.")
        failures += 1
        continue

    # Fetch or reuse the series
    try:
        series = av.fetch_series(sym, key_idx)
        print(f"   ✓ Downloaded/cached {sym} with {len(series)} trading days [{series.index.min().date()} .. {series.index.max().date()}]")
    except Exception as e:
        print(f"   ! Fetch failed for {sym}: {e}")
        failures += 1
        continue

    # Compute T-block by rules
    T_date, T_price, T1_date, T1_price, T2_date, T2_price, T3_date, T3_price = compute_T_block(series, rdate)

    if T_date is None or T_price is None:
        print("   ! Could not find a T date (no trade on/ before Result Date). Skipping row.")
        failures += 1
        continue

    # Counters (exact vs previous)
    if pd.Timestamp(rdate.normalize()) == pd.Timestamp(T_date.normalize()):
        exact_hits += 1
        print(f"   -> Exact match on Result Date. T={_fmt_date(T_date)} Close={T_price:.2f}")
    else:
        prev_hits += 1
        print(f"   -> No trade on Result Date; used previous trading day.")
        print(f"      T={_fmt_date(T_date)} Close={T_price:.2f}")

    # Update row in df (dates as DD-MM-YYYY strings, prices as floats)
    df.at[ridx, "_T_date"]               = _fmt_date(T_date)
    df.at[ridx, "Closing Price (T)"]     = round(T_price, 2)

    df.at[ridx, "_T1_date"]              = _fmt_date(T1_date)
    df.at[ridx, "Closing Price (T+1)"]   = round(T1_price, 2) if T1_price is not None else None

    df.at[ridx, "_T2_date"]              = _fmt_date(T2_date)
    df.at[ridx, "Closing Price (T+2)"]   = round(T2_price, 2) if T2_price is not None else None

    df.at[ridx, "_T3_date"]              = _fmt_date(T3_date)
    df.at[ridx, "Closing Price (T+3)"]   = round(T3_price, 2) if T3_price is not None else None

    # Row-level recap
    print("      T1=", _fmt_date(T1_date), f"Close={round(T1_price,2):.2f}" if T1_price is not None else "Close=NA")
    print("      T2=", _fmt_date(T2_date), f"Close={round(T2_price,2):.2f}" if T2_price is not None else "Close=NA")
    print("      T3=", _fmt_date(T3_date), f"Close={round(T3_price,2):.2f}" if T3_price is not None else "Close=NA")

    updated += 1

# Put the updated sheet back
sheets["all_prices"] = df

# -----------------------------
# Write out test_data_4.xlsx (preserve other sheets)
# -----------------------------
print(f"\nWriting updated workbook: {OUTPUT_XLSX}")
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    # Keep original sheet order
    for name in xls.sheet_names:
        sheets[name].to_excel(writer, index=False, sheet_name=name)

print("\n=== Summary ===")
print(f"Rows needing fill                : {n_to_fill}")
print(f"Rows updated                     : {updated}")
print(f" - T matched exact Result Date   : {exact_hits}")
print(f" - T used previous trading day   : {prev_hits}")
print(f"Failures / still blank           : {failures}")
print(f"Output file                      : {Path(OUTPUT_XLSX).resolve()}")


Reading input workbook: test_data_3.xlsx
Total rows in 'all_prices': 585
Rows needing prices (blank _T_date): 43

[1/43] TCS | symbol=TCS.BSE | Result Date=11-07-2024 | using key K1
   ✓ Downloaded/cached TCS.BSE with 5095 trading days [2005-01-03 .. 2025-08-12]
   -> Exact match on Result Date. T=11-07-2024 Close=3922.70
      T1= 12-07-2024 Close=4184.90
      T2= 15-07-2024 Close=4168.00
      T3= 16-07-2024 Close=4175.35

[2/43] INFY | symbol=INFY.BSE | Result Date=18-07-2024 | using key K1
   ✓ Downloaded/cached INFY.BSE with 5097 trading days [2005-01-03 .. 2025-08-12]
   -> Exact match on Result Date. T=18-07-2024 Close=1759.15
      T1= 19-07-2024 Close=1792.85
      T2= 22-07-2024 Close=1810.55
      T3= 23-07-2024 Close=1836.95

[3/43] HCLTECH | symbol=HCLTECH.BSE | Result Date=12-07-2024 | using key K1
   ✓ Downloaded/cached HCLTECH.BSE with 5077 trading days [2005-01-03 .. 2025-08-12]
   -> Exact match on Result Date. T=12-07-2024 Close=1560.40
      T1= 15-07-2024 Close=15